In [ ]:
!pip install google-generativeai

import json
import google.generativeai as genai
from google.colab import files, drive
from datetime import datetime


print("✅ Setup complete")

In [ ]:
from google.colab import userdata
API_KEY = ""

# Configure Gemini
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-3-flash-preview')

# Test connection
response = model.generate_content("Say 'API is ready'")
print(response.text)

API is ready


In [ ]:
# Install pypdf if not already installed
!pip install pypdf

import pypdf # Import the library

# Upload the cleaned story from previous agent
print("📤 Upload your story text file:")
uploaded = files.upload()

# Get the filename
story_file = list(uploaded.keys())[0]
print(f"✅ Uploaded: {story_file}")

# Read the story, now specifically for PDF files
story_text = ""
if story_file.endswith('.pdf'):
    with open(story_file, 'rb') as f: # Open in binary mode for PDF
        reader = pypdf.PdfReader(f)
        for page in reader.pages:
            story_text += page.extract_text() + "\n"
else:
    # If not a PDF, try reading as a standard text file
    with open(story_file, 'r', encoding='utf-8') as f:
        story_text = f.read()

print(f"✓ Story length: {len(story_text)} characters")
print(f"✓ Word count: {len(story_text.split())}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 17.5 MB/s eta 0:00:00
📤 Upload your story text file:


Saving cleaned_story.txt to cleaned_story (1).txt
✅ Uploaded: cleaned_story (1).txt
✓ Story length: 11380 characters
✓ Word count: 2191


In [ ]:
import json
import re

def extract_characters(story_text, output_file="characters.json"):
    print("Analyzing story for characters...")

    prompt = """
You are a STRICT character and voice profiling system for audiobook generation.

Your role is to read the FULL story and extract ALL relevant speaking characters,
then generate CONSISTENT, NON-HALLUCINATED voice embeddings suitable for TTS systems.

This output will be directly used for voice synthesis. Accuracy and consistency are critical.

==================================================
PRIMARY OBJECTIVE
==================================================

For each important character:
1. Identify the character
2. Understand how they GENERALLY speak across the story
3. Generate a stable voice description (NOT situation-based)
4. Provide one real dialogue line as reference

Also include a NARRATOR entry.

==================================================
CHARACTER SELECTION RULES (STRICT)
==================================================

Include ONLY characters that meet at least ONE:
1. Have dialogue in the story
2. Appear multiple times
3. Influence the story progression

Exclude:
- unnamed background characters
- one-time mentions without importance
- symbolic or abstract entities

==================================================
CORE EXTRACTION RULES
==================================================

1. ZERO HALLUCINATION
- DO NOT invent traits
- DO NOT invent age, accent, pitch, tone, personality, or speaking style
- ONLY use information inferable from:
  - dialogue
  - repeated behavior
  - explicit descriptions

If uncertain -> OMIT that detail

--------------------------------------------------

2. GENERAL VOICE ONLY (CRITICAL)
- Describe how the character speaks OVERALL across the story
- DO NOT include temporary or situation-based delivery such as:
  - whispering to stay hidden
  - shouting during battle
  - crying in one emotional scene

Instead capture the stable baseline voice:
- likely age category if inferable
- likely gender if clear
- pitch tendency
- pace tendency
- overall tone
- speaking style
- accent ONLY if clearly supported by the text

--------------------------------------------------

3. AGE RULE
- Use GENERAL categories ONLY
- Allowed examples:
  - "young boy"
  - "young girl"
  - "teenage boy"
  - "teenage girl"
  - "young adult male"
  - "young adult female"
  - "adult male"
  - "adult female"
  - "middle-aged man"
  - "middle-aged woman"
  - "elderly man"
  - "elderly woman"

- NEVER use exact ages or age ranges like:
  - "19 years old"
  - "around 45"
  - "70+"

--------------------------------------------------

4. DESCRIPTION RULE
Each character description should be a concise TTS-oriented voice embedding.

Include ONLY supported traits such as:
- age category
- gender if clear
- pitch (high / medium / deep)
- pace (slow / medium / fast)
- tone/personality (calm, nervous, confident, cold, warm, playful, stern, etc.)
- speaking style (measured, expressive, blunt, formal, reactive, monotone, etc.)
- accent ONLY if clearly indicated

Do NOT include:
- scene-specific delivery
- physical appearance unless it clearly helps infer the voice
- invented dramatic details
- poetic or cinematic phrasing

--------------------------------------------------

5. REFERENCE TEXT (MANDATORY)
- Must be an EXACT line spoken by the character
- Extract directly from the story
- Do NOT paraphrase
- Do NOT generate new dialogue
- Keep it to one suitable spoken line

--------------------------------------------------

6. NARRATOR RULE (IMPORTANT)
- ALWAYS include a "narrator" entry
- The narrator should represent the storytelling voice used for non-dialogue text
- Narrator description should be neutral, clear, and suitable for continuous narration
- narrator ref_text must be a real narration line from the story, NOT dialogue

==================================================
OUTPUT FORMAT (STRICT JSON ONLY)
==================================================

{{
  "characters": {{
    "narrator": {{
      "name": "Narrator",
      "description": "Neutral voice, adult, clear and steady tone, medium pace, balanced narration style",
      "ref_text": "Exact narration line from the story"
    }},
    "character_id": {{
      "name": "Character name exactly as in text",
      "description": "TTS-oriented general voice embedding based only on supported evidence",
      "ref_text": "Exact dialogue line from the story"
    }}
  }}
}}

==================================================
EXAMPLES (FOLLOW THIS STYLE STRICTLY)
==================================================

{{
  "characters": {{
    "narrator": {{
      "name": "Narrator",
      "description": "Neutral voice, adult, clear and steady tone, medium pace, balanced and expressive narration style",
      "ref_text": "The wind moved silently through the ancient trees as the village slept under a restless sky."
    }},
    "wizard": {{
      "name": "Wise Old Wizard",
      "description": "Elderly male voice, deep and slightly raspy, slow and measured speech, calm and thoughtful tone, speaks with quiet authority",
      "ref_text": "I have watched over this realm for many years. The ancient magic still flows, though few remember its true power."
    }},
    "villain": {{
      "name": "Dark Lord",
      "description": "Adult male voice, deep and controlled, slow deliberate pacing, cold and menacing tone, emotionally restrained but intense delivery",
      "ref_text": "You think you can defeat me? Your courage will fade long before your strength does."
    }},
    "heroine": {{
      "name": "Young Heroine",
      "description": "Young adult female voice, clear and strong, medium pace, determined and confident tone, emotionally expressive but controlled",
      "ref_text": "I am not afraid anymore. This is where I stand, and I will not turn back."
    }},
    "sidekick": {{
      "name": "Companion",
      "description": "Young adult male voice, slightly higher pitch, fast-paced speech, nervous but energetic tone, expressive and reactive speaking style",
      "ref_text": "Wait, wait... are we really doing this? This feels like a very bad idea."
    }}
  }}
}}

==================================================
FINAL VALIDATION (MANDATORY)
==================================================

Before output:
- Remove characters with only one insignificant appearance
- Ensure NO hallucinated attributes
- Ensure ALL descriptions reflect GENERAL voice, not situations
- Ensure ref_text is EXACTLY from story
- Ensure narrator is present
- Ensure valid JSON only
- Do NOT wrap output in markdown
- Do NOT add explanation text before or after JSON

==================================================
INPUT STORY
==================================================

{story_text}
""".format(story_text=story_text)

    response = model.generate_content(prompt)

    if not hasattr(response, "text") or not response.text:
        raise ValueError("Model returned an empty response.")

    json_text = response.text.strip()

    # Remove accidental markdown fences if added by the model
    json_text = re.sub(r"^```json\s*", "", json_text, flags=re.IGNORECASE)
    json_text = re.sub(r"^```\s*", "", json_text)
    json_text = re.sub(r"\s*```$", "", json_text)

    try:
        characters_data = json.loads(json_text)
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse model output as JSON.\n\nModel output:\n{json_text}") from e

    if "characters" not in characters_data or not isinstance(characters_data["characters"], dict):
        raise ValueError("JSON does not contain a valid 'characters' dictionary.")

    cleaned_characters = {}

    for char_id, char_data in characters_data["characters"].items():
        if not isinstance(char_data, dict):
            continue

        name = str(char_data.get("name", "")).strip()
        description = str(char_data.get("description", "")).strip()
        ref_text = str(char_data.get("ref_text", "")).strip()

        if not name:
            name = char_id.replace("_", " ").title()

        if not description:
            description = "Neutral voice, adult, clear tone, medium pace"

        cleaned_characters[char_id] = {
            "name": name,
            "description": description,
            "ref_text": ref_text
        }

    characters_data["characters"] = cleaned_characters
    characters_data["total_characters"] = len(cleaned_characters)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(characters_data, f, indent=2, ensure_ascii=False)

    print(f"Extracted {characters_data['total_characters']} character profiles")
    return characters_data

In [ ]:
# Extract characters
characters_data = extract_characters(story_text, "characters.json")

if characters_data:
    print("\n" + "=" * 60)
    print("CHARACTER EXTRACTION COMPLETE")
    print("=" * 60)

    total_chars = characters_data.get("total_characters", len(characters_data.get("characters", {})))
    print(f"\nTotal character profiles found: {total_chars}")

    if "characters" in characters_data and characters_data["characters"]:
        print("\nCharacter Voice Profiles:")
        print("-" * 60)

        char_items = list(characters_data["characters"].items())

        for i, (char_id, char) in enumerate(char_items[:5], 1):
            print(f"\n{i}. ID: {char_id}")
            print(f"   Name: {char.get('name', 'Unknown')}")
            print(f"   Description: {char.get('description', '')}")
            print(f"   Ref Text: {char.get('ref_text', '')[:200]}")

        if len(char_items) > 5:
            print(f"\n... and {len(char_items) - 5} more characters")

    print("\nSaved to characters.json")

Analyzing story for characters...
Extracted 3 character profiles

CHARACTER EXTRACTION COMPLETE

Total character profiles found: 3

Character Voice Profiles:
------------------------------------------------------------

1. ID: narrator
   Name: Narrator
   Description: Neutral voice, adult, clear and steady tone, medium pace, balanced and observant narration style
   Ref Text: Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do.

2. ID: alice
   Name: Alice
   Description: Young girl voice, clear and expressive, medium pace, inquisitive and polite tone, thoughtful and imaginative speaking style, uses formal vocabulary for her age
   Ref Text: I wonder if I shall fall right through the earth!

3. ID: white_rabbit
   Name: White Rabbit
   Description: Adult male voice, medium pitch, fast-paced and frantic speech, anxious and hurried tone, reactive and repetitive delivery
   Ref Text: Oh my ears and whiskers, how late it’s getting !

Sav

In [ ]:
# Download the characters.json file
from google.colab import files

files.download("characters.json")
print("✅ characters.json downloaded successfully!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ characters.json downloaded successfully!


In [ ]:
# ===== AGENT 2 TEST CELL =====

test_story = """
Arjun stepped into the dark forest. The wind whispered through the trees.

"I have a bad feeling about this," Arjun said.

A shadow moved behind him.

"Who's there?" he shouted.

From somewhere deeper in the forest, a calm voice replied,
"You should not have come here."

Arjun tightened his grip on the lantern and looked around.
The trees seemed to lean closer as the silence thickened.

A woman emerged from the mist, her eyes fixed on him.
"If you value your life, turn back now," she said.

But Arjun stood his ground.
"I didn’t come this far to run."
"""

print("🔹 Testing Agent 2 with hardcoded story text...\n")

characters_data = extract_characters(test_story, "test_characters.json")

print("\n" + "=" * 60)
print("✅ AGENT 2 OUTPUT")
print("=" * 60)

print(json.dumps(characters_data, indent=2, ensure_ascii=False))

print("\n💾 Saved to test_characters.json")

🔹 Testing Agent 2 with hardcoded story text...

Analyzing story for characters...
Extracted 3 character profiles

✅ AGENT 2 OUTPUT
{
  "characters": {
    "narrator": {
      "name": "Narrator",
      "description": "Neutral voice, adult, clear and steady tone, medium pace, balanced narration style",
      "ref_text": "Arjun stepped into the dark forest."
    },
    "arjun": {
      "name": "Arjun",
      "description": "Adult male voice, medium pitch, medium pace, determined and reactive tone, expressive speaking style",
      "ref_text": "I have a bad feeling about this."
    },
    "woman": {
      "name": "Woman",
      "description": "Adult female voice, medium pitch, slow and measured pace, calm and authoritative tone, steady speaking style",
      "ref_text": "If you value your life, turn back now."
    }
  },
  "total_characters": 3
}

💾 Saved to test_characters.json


In [ ]:
# ===== SAVE + DOWNLOAD AGENT 2 OUTPUT =====

import json
from google.colab import files

if "characters_data" not in globals():
    raise ValueError("characters_data not found. Run the Agent 2 test cell first.")

output_filename = "characters_data.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(characters_data, f, indent=2, ensure_ascii=False)

print(f"✅ Saved Agent 2 output to {output_filename}")

files.download(output_filename)

print(f"✅ Download started for {output_filename}")

✅ Saved Agent 2 output to characters_data.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started for characters_data.json
